# Joins and Aggregations in SQL


## Why Joins Matter

In real systems, data is NEVER in one table.

Example:
- Users table
- Orders table

To answer:
👉 "How much did each user spend?"

You MUST join tables.

👉 Joins = core of SQL interviews


## Prerequisites

- PostgreSQL running (see [`README.md`](README.md)).
- This notebook **drops and recreates** `users` and `orders` so it stays consistent even if you completed `01_sql_basics.ipynb` first (different `users` shape).


## Connect and create tables


In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="de_course",
    user="admin",
    password="admin",
)

cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS orders CASCADE")
cur.execute("DROP TABLE IF EXISTS users CASCADE")

cur.execute(
    """
CREATE TABLE IF NOT EXISTS users (
    id SERIAL PRIMARY KEY,
    name TEXT
)
"""
)

cur.execute(
    """
CREATE TABLE IF NOT EXISTS orders (
    id SERIAL PRIMARY KEY,
    user_id INT,
    amount INT
)
"""
)

conn.commit()


## Insert data


In [ ]:
cur.execute(
    """
INSERT INTO users (name)
VALUES ('Amar'), ('Ravi'), ('Neha')
"""
)

cur.execute(
    """
INSERT INTO orders (user_id, amount)
VALUES
(1, 100),
(1, 200),
(2, 150),
(3, 300)
"""
)

conn.commit()


## INNER JOIN


In [ ]:
cur.execute(
    """
SELECT users.name, orders.amount
FROM users
INNER JOIN orders
ON users.id = orders.user_id
"""
)

print(cur.fetchall())


## LEFT JOIN


In [ ]:
cur.execute(
    """
SELECT users.name, orders.amount
FROM users
LEFT JOIN orders
ON users.id = orders.user_id
"""
)

print(cur.fetchall())


## Aggregation


In [ ]:
cur.execute(
    """
SELECT user_id, SUM(amount) as total_spent
FROM orders
GROUP BY user_id
"""
)

print(cur.fetchall())


## Join + aggregation


In [ ]:
cur.execute(
    """
SELECT users.name, SUM(orders.amount) as total_spent
FROM users
JOIN orders ON users.id = orders.user_id
GROUP BY users.name
"""
)

print(cur.fetchall())


## HAVING (IMPORTANT)


In [ ]:
cur.execute(
    """
SELECT users.name, SUM(orders.amount) as total_spent
FROM users
JOIN orders ON users.id = orders.user_id
GROUP BY users.name
HAVING SUM(orders.amount) > 150
"""
)

print(cur.fetchall())


## Real interview pattern


In [ ]:
cur.execute(
    """
SELECT users.name, COUNT(orders.id) as total_orders
FROM users
LEFT JOIN orders ON users.id = orders.user_id
GROUP BY users.name
"""
)

print(cur.fetchall())


## Practice

1. Find user with highest spending
2. Find users with no orders
3. Find average order per user


## Assignment

Extend schema:

Add `products` table:
- `id`
- `name`
- `price`

Modify `orders`:
- `product_id`

Now answer:

1. Total revenue per product
2. Most popular product
3. User who spent the most


## Interview Questions

1. Difference between INNER and LEFT JOIN?
2. What is GROUP BY?
3. Difference between WHERE and HAVING?
4. How do you find missing data?


## What you just learned

- How tables **connect**
- How to **aggregate** business metrics
- How to answer **real questions** from relational data

👉 Daily work for analysts and data engineers.


---

**Next:** **Window functions** — interview-heavy, powerful analytics patterns.


## Optional: close connection


In [ ]:
cur.close()
conn.close()
print("Connection closed.")


## Your tasks

- [ ] PostgreSQL up; run **Connect and create tables** through **Real interview pattern** in order.
- [ ] **Practice:** highest spender; users with no orders (hint: `LEFT JOIN` + `WHERE orders.id IS NULL`); average order size per user.
- [ ] **Assignment:** add `products`, add `product_id` on `orders`, run the three business questions (consider `JOIN` + `GROUP BY` / subqueries).
- [ ] Drill interview answers; explain **`COUNT(orders.id)` vs `COUNT(*)`** with `LEFT JOIN`.


## Shared dataset usage (`resources/datasets/`)

Use reusable files for joins and aggregation drills:

```sql
-- Example load path (adjust for your SQL environment)
-- users and orders are in JSON; transactions are in CSV.
-- transactions.csv is easiest for quick SQL practice:

-- PostgreSQL example:
-- CREATE TABLE transactions (user_id INT, amount INT, category TEXT);
-- \copy transactions FROM 'resources/datasets/transactions.csv' WITH (FORMAT csv, HEADER true);
```

Suggested files:
- `resources/datasets/users.json`
- `resources/datasets/orders.json`
- `resources/datasets/transactions.csv`